# Experimento 2B: Ilusión de la Mayoría (Redes Sintéticas)

Este notebook replica el experimento de la Figura 2 del paper de la Ilusión de la Mayoría en redes sintéticas.

## 1. Configuración del Entorno


In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import random

N = 10000
p_active = 0.05
alphas = [2.1, 2.4]

## 2. Generación de Red Libre de Escala


In [ ]:
def generate_scale_free_network(N, alpha):
    while True:
        s = np.random.zipf(alpha, N)
        s = np.clip(s, 1, int(np.sqrt(N))) # Limitar hub maximo
        if sum(s) % 2 != 0:
            s[0] += 1
        G = nx.configuration_model(s)
        G = nx.Graph(G)
        G.remove_edges_from(nx.selfloop_edges(G))
        components = sorted(nx.connected_components(G), key=len, reverse=True)
        if len(components) == 0: continue
        G = G.subgraph(components[0]).copy()
        if len(G) > N * 0.8:
            break
    return G


In [ ]:
def assign_initial_attributes(G, p_active):
    nodes = list(G.nodes())
    num_active = int(len(nodes) * p_active)
    nx.set_node_attributes(G, 0, 'x')
    active_nodes = random.sample(nodes, num_active)
    for n in active_nodes:
        G.nodes[n]['x'] = 1

def calculate_rhokx(G):
    k = np.array([d for n, d in G.degree()])
    x = np.array([d['x'] for n, d in G.nodes(data=True)])
    sigma_k = np.std(k)
    sigma_x = np.std(x)
    if sigma_x == 0 or sigma_k == 0:
        return 0.0
    P_x1 = np.mean(x)
    mean_k = np.mean(k)
    k_x1 = k[x == 1]
    mean_k_x1 = np.mean(k_x1) if len(k_x1) > 0 else 0
    return (P_x1 / (sigma_x * sigma_k)) * (mean_k_x1 - mean_k)


## 3. Algoritmos de Manipulación


In [ ]:
def attribute_swapping(G, target_rho, max_steps=50000, tol=0.01):
    active = [n for n, attr in G.nodes(data=True) if attr['x'] == 1]
    inactive = [n for n, attr in G.nodes(data=True) if attr['x'] == 0]
    degree_dict = dict(G.degree())
    current_rho = calculate_rhokx(G)
    for step in range(max_steps):
        if step > 0 and step % 10000 == 0:
            print(f"  [Attribute Swapping] Paso {step}/{max_steps} | rho actual: {current_rho:.4f}")
        if abs(current_rho - target_rho) < tol:
            break
        v_active = random.choice(active)
        v_inactive = random.choice(inactive)
        k_active = degree_dict[v_active]
        k_inactive = degree_dict[v_inactive]
        if target_rho > current_rho and k_inactive > k_active:
            G.nodes[v_active]['x'] = 0
            G.nodes[v_inactive]['x'] = 1
            active.remove(v_active)
            active.append(v_inactive)
            inactive.remove(v_inactive)
            inactive.append(v_active)
            current_rho = calculate_rhokx(G)
        elif target_rho < current_rho and k_active > k_inactive:
            G.nodes[v_active]['x'] = 0
            G.nodes[v_inactive]['x'] = 1
            active.remove(v_active)
            active.append(v_inactive)
            inactive.remove(v_inactive)
            inactive.append(v_active)
            current_rho = calculate_rhokx(G)
    return current_rho

def edge_rewiring(G, target_r, max_steps=20000, tol=0.01):
    edges = list(G.edges())
    current_r = nx.degree_assortativity_coefficient(G)
    for step in range(max_steps):
        if step > 0 and step % 10000 == 0:
            print(f"  [Edge Rewiring] Paso {step}/{max_steps} | r actual: {current_r:.4f}")
        if abs(current_r - target_r) < tol:
            break
        e1, e2 = random.sample(edges, 2)
        u, v = e1
        w, z = e2
        if len({u, v, w, z}) < 4:
            continue
        if not G.has_edge(u, w) and not G.has_edge(v, z):
            G.remove_edge(u, v)
            G.remove_edge(w, z)
            G.add_edge(u, w)
            G.add_edge(v, z)
            new_r = nx.degree_assortativity_coefficient(G)
            if abs(new_r - target_r) <= abs(current_r - target_r):
                current_r = new_r
                edges.remove(e1)
                edges.remove(e2)
                edges.append((u, w))
                edges.append((v, z))
            else:
                G.remove_edge(u, w)
                G.remove_edge(v, z)
                G.add_edge(u, v)
                G.add_edge(w, z)
    return current_r


## 4. Medición y Simulación


In [ ]:
def calculate_illusion(G):
    count = 0
    nodos_con_vecinos = [n for n in G.nodes() if G.degree(n) > 0]
    for n in nodos_con_vecinos:
        vecinos = list(G.neighbors(n))
        vecinos_activos = sum(1 for v in vecinos if G.nodes[v]['x'] == 1)
        if vecinos_activos / len(vecinos) > 0.5:
            count += 1
    return count / len(nodos_con_vecinos) if nodos_con_vecinos else 0

results = {}
target_rhos = np.linspace(0.1, 0.8, 6)
target_assortativities = [0.15, 0.0, -0.15]
print(f"Iniciando simulación para alphas={alphas}...")

for alpha in alphas:
    print(f"\n--- Alpha = {alpha} ---")
    G_base = generate_scale_free_network(N, alpha)
    results[alpha] = []
    for target_r in target_assortativities:
        print(f"\n► Asortatividad objetivo r={target_r:.2f}")
        G_r = G_base.copy()
        actual_r = edge_rewiring(G_r, target_r=target_r, max_steps=20000)
        print(f"  Terminado r={actual_r:.3f}")
        curve_data = []
        for trho in target_rhos:
            print(f"  >> rho objetivo={trho:.2f}")
            G_sim = G_r.copy()
            assign_initial_attributes(G_sim, p_active)
            actual_rho = attribute_swapping(G_sim, target_rho=trho, max_steps=25000)
            illusion = calculate_illusion(G_sim)
            curve_data.append((actual_rho, illusion))
            print(f"     Final rho={actual_rho:.3f}, ilusión={illusion:.3f}")
        results[alpha].append({'r_actual': actual_r, 'data': curve_data})
print("\nSimulación completada.")


## 5. Visualización (Figura 2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, alpha in enumerate(alphas):
    ax = axes[i]
    for curve in results[alpha]:
        r_str = f"r ≈ {curve['r_actual']:.2f}"
        data = sorted(curve['data'], key=lambda d: d[0])
        x = [d[0] for d in data]
        y = [d[1] for d in data]
        ax.plot(x, y, marker='o', label=r_str)
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('Correlación Grado-Atributo ($ρ_{kx}$)')
    ax.set_ylabel('Fracción de Ilusión ($P_{>1/2}$)')
    ax.legend(title='Asortatividad', loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlim(-0.05, 1.05)

plt.suptitle('Ilusión de la Mayoría en Redes Sintéticas', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('figura_2_replicada.png', dpi=300, bbox_inches='tight')
plt.show()


## Conclusión

**¿Por qué, cuando los nodos de alto grado (hubs) se vuelven activos, la percepción general de la red cambia drásticamente?**

En una red libre de escala, muy pocos nodos concentran una inmensa cantidad de conexiones. Al activar estos *hubs* (forzando una alta correlación $\rho_{kx}$ a través de *Attribute Swapping*), permitimos que esta minoría adopte el atributo (que representa globalmente del 5% de la red).

Debido a su elevadísima conectividad, los *hubs* se encuentran indudablemente en la vecindad inmediata de la enorme mayoría de los nodos de bajo grado en la periferia. Al examinar localmente su entorno, estos nodos periféricos enfrentan frecuentemente que sus contados vecinos son *hubs* activos. Esto infla la proporción local de vecinos activos por encima del $50\%$, desatando en gran parte a la Ilusión de la Mayoría.

Además, la resiliencia a esta distorsión depende fuertemente de la **asortatividad ($r_{kk}$)**:
- Si la red es **desasortativa** ($r < 0$), los *hubs* se conectan mayoritariamente con nodos periféricos de bajo grado. Este despliegue maximiza la exposición de la red a la influencia del hub, disparando la ilusión velozmente.
- Si la red es **asortativa** ($r > 0$), los *hubs* tienden a enlazarse entre sí aislando la conectividad e interacciones entre elites de alta centralidad (*rich-club*). Aquí, el atributo minoritario en lugar de colonizar la periferia, se atora y recicla solo en el núcleo cerrado, demorando y minimizando significativamente el impacto sobre la percepción transversal de la red.